# Jun-2026 LAPD Langmuir IV — one-position inspection

Walks the full IV workflow for **one probe position** so traces and intermediate
results can be checked and knobs tuned before any batch run / saving.

Reader + analysis helpers live in `Jun2026_IV.py`; this notebook just drives them.

**This stage saves nothing** — no `.npz`, no plot files, no batch loop.

Workflow: configure → discover channels → positions → read raw → detect sweeps
→ reshape + smooth → analyze one trace → repeat per probe.


## 1. Configure — set the run file and knobs

In [1]:
import importlib
get_ipython().run_line_magic("matplotlib", "qt")
# get_ipython().run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt

import Jun2026_IV as jiv
importlib.reload(jiv)   # re-run this cell after editing Jun2026_IV.py

from data_analysis.io import open_lapd
from data_analysis.plasma.langmuir import find_sweep_indices, reshape_IV, analyze_IV
from scipy.ndimage import gaussian_filter1d

# >>> SET THIS to the run you want to inspect (leave knobs in Jun2026_IV.py) <<<
ifn = r"D:\data\LAPD\jun2026-jia\25-He-800G-bias40V-LP-p29-line_2026-06-12.hdf5"

# Knobs live in Jun2026_IV.py and apply everywhere (RESISTOR, Aprobe, I_SIGN,
# SCOPE_NAME / I_CHAN / V_CHAN override).  To retune: edit that file, then re-run
# the cell above (it reloads the module).  Current sign: I_SIGN = -1 for this
# experiment (electron branch up after reshape_IV's fixed extra negation).
print(f"RESISTOR = {jiv.RESISTOR}, Aprobe = {jiv.Aprobe}, I_SIGN = {jiv.I_SIGN}")
print(f"override: SCOPE_NAME={jiv.SCOPE_NAME}, I_CHAN={jiv.I_CHAN}, V_CHAN={jiv.V_CHAN}")

assert ifn, "Set `ifn` to a run file first."
run = open_lapd(ifn)
print("backend:", run.backend)

RESISTOR = 25.0, Aprobe = 0.002, I_SIGN = -1
override: SCOPE_NAME=None, I_CHAN=None, V_CHAN=None
backend: pydaq


## 2. Discover I/V channels (all complete probe pairs)

Auto-detects the LP scope group and, from each channel's `description`, groups
channels by tip into complete I+V pairs.  **Every** complete pair is collected
(both probes analyzed); tips missing a channel are flagged and left blank, and
fixed-bias ion-saturation (I-only) tips are not paired.

In [3]:
scope_name = jiv.SCOPE_NAME if jiv.SCOPE_NAME is not None else jiv.find_lp_scope(ifn)
pairs, incomplete = jiv.discover_lp_channels(ifn, scope_name)

# tips to analyze (override forces a single tip; else all complete pairs)
if jiv.I_CHAN is not None and jiv.V_CHAN is not None:
    tips = {"override": {"I": jiv.I_CHAN, "V": jiv.V_CHAN}}
else:
    tips = dict(pairs)

print("\nTips that will be analyzed:")
for tip, ch in tips.items():
    print(f"  tip {tip}: I={ch['I']}, V={ch['V']}")
if incomplete:
    print("Tips left BLANK (incomplete):", list(incomplete))


LP scope 'scope' channel descriptions:
  C1: 'I@P33, L'  -> quantity=I, tip=L
  C2: 'V@P33, L'  -> quantity=V, tip=L
  C3: 'I@P33, R'  -> quantity=I, tip=R
  C4: 'V@P33, R'  -> quantity=V, tip=R

Complete I+V tips (will be analyzed):
  tip L: I=C1, V=C2
  tip R: I=C3, V=C4

Tips that will be analyzed:
  tip L: I=C1, V=C2
  tip R: I=C3, V=C4


## 3. Positions — pick one to inspect

In [4]:
pos_array, xpos, ypos, npos, nshot = jiv.read_lp_positions(ifn)

pos_index = npos//2
print(f"\nInspecting position index {pos_index} of {npos} "
      f"(x={xpos[pos_index] if pos_index < len(xpos) else '?'} cm), {nshot} shots")

Using motion group: '<Hermes>    p29_LP'
Positions: 61 unique (x: 61, y: 1), 5 shots/position, 305 total shots.

Inspecting position index 30 of 61 (x=0.0 cm), 5 shots


## 4. Read raw I, V at this position

Shot-averaged V and per-shot I (scaled per the knobs).  Inspect the raw traces:
V should show the sweep ramps; I should be the probe current.

In [9]:
# pick which tip to look at in cells 4-7 (loop over `tips` for both probes)
tip = next(iter(tips))
I_chan, V_chan = tips[tip]["I"], tips[tip]["V"]
print(f"tip {tip}: I={I_chan}, V={V_chan}")

tarr, Vpos, Ipos = jiv.get_IV_at_position(
    run, scope_name, I_chan, V_chan, npos, nshot, pos_index)
print("Vpos:", Vpos.shape, " Ipos:", Ipos.shape, " tarr:", tarr.shape)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(tarr * 1e3, Vpos)
ax[0].set_ylabel("V probe [V]")
ax[0].set_title(f"Raw traces — pos {pos_index}, tip {tip}")
for s in range(Ipos.shape[0]):
    ax[1].plot(tarr * 1e3, Ipos[s], lw=0.6, alpha=0.6)
ax[1].plot(tarr * 1e3, Ipos.mean(0), "k", lw=1.2, label="shot mean")
ax[1].set_ylabel("I [a.u.]"); ax[1].set_xlabel("t [ms]")
ax[1].legend(loc="upper right")
plt.tight_layout(); plt.show()

tip L: I=C1, V=C2
Vpos: (100001,)  Ipos: (5, 100001)  tarr: (100001,)


## 5. Detect sweeps

`find_sweep_indices` locates each voltage ramp.  Tune `padding` if boundaries
look wrong; the overlay shows where each sweep starts/stops.

In [8]:
padding = 10   # <<< tune sweep boundary padding

start_ls, stop_ls = find_sweep_indices(Vpos, padding=padding)
print(f"Detected {len(start_ls)} sweeps")

plt.figure(figsize=(11, 3.5))
plt.plot(tarr * 1e3, Vpos, lw=0.8)
for a, b in zip(start_ls, stop_ls):
    plt.axvspan(tarr[a] * 1e3, tarr[b] * 1e3, color="orange", alpha=0.2)
plt.xlabel("t [ms]"); plt.ylabel("V probe [V]")
plt.title(f"Sweep windows ({len(start_ls)} found)")
plt.tight_layout(); plt.show()

Detected 25 sweeps


## 6. Reshape into per-sweep traces + smooth

`reshape_IV` standardizes sweep length and trims edges; current is smoothed.
Plot a few I–V traces to confirm shape and sign (electron branch should rise at
high V — flip `I_SIGN` in `Jun2026_IV.py` if inverted).

In [7]:
trim_percent = 5    # <<< edge trim
smooth_sigma = 10   # <<< current smoothing

V_rs, I_rs = reshape_IV(Vpos[None, :], Ipos[None, :, :], start_ls, stop_ls, trim_percent)
I_rs = gaussian_filter1d(I_rs, smooth_sigma, axis=-1)

V_sweeps = V_rs[0]            # (nsweep, L)
I_sweeps = I_rs[0]           # (nshot, nsweep, L)
print("V_sweeps:", V_sweeps.shape, " I_sweeps:", I_sweeps.shape)

plt.figure(figsize=(8, 5))
for k in range(V_sweeps.shape[0]):
    plt.plot(V_sweeps[k], I_sweeps[:, k, :].mean(0), label=f"sweep {k}")
plt.xlabel("V probe [V]"); plt.ylabel("I [a.u.]")
plt.title(f"I-V traces (shot-averaged) - pos {pos_index}, tip {tip}")
plt.legend(); plt.tight_layout(); plt.show()

Sweep length: 1974 points (0 partial sweep(s) dropped).
Trimming 5% (98 points) from each end.
Final sweep length stacked: 1778 points.
V_sweeps: (25, 1778)  I_sweeps: (5, 25, 1778)


## 7. Analyze one I–V trace

`analyze_IV(V, I, plot=True)` extracts Vp / Te / ne and shows the fit.  Pick a
sweep index and shot to inspect the fit quality.

In [12]:
sweep_idx = 1                  # <<< which sweep
I_trace = I_sweeps[:, sweep_idx, :].mean(0)   # shot-averaged current for this sweep
V_trace = V_sweeps[sweep_idx]

Vp, Te, ne = analyze_IV(V_trace, I_trace, plot=True)
print(f"Vp = {Vp:.2f} V,  Te = {Te:.2f} eV,  ne = {ne:.3e} cm^-3")

Vp = 1.02 V,  Te = 0.77 eV,  ne = 3.665e+12 cm^-3


## 8. Both probes

Loop the read → detect → reshape → analyze steps over every complete-pair tip so
both probes are inspected at this position.  Missing probes stay blank.

In [ ]:
# analyze_tip_at_position bundles the read->detect->reshape->smooth->analyze
# steps walked manually in cells 4-7, so both probes use the same pipeline.
for t, ch in tips.items():
    Vp, Te, ne = jiv.analyze_tip_at_position(
        run, scope_name, ch["I"], ch["V"], npos, nshot, pos_index, sweep_idx,
        padding=padding, trim_percent=trim_percent, smooth_sigma=smooth_sigma)
    print(f"tip {t} (I={ch['I']}, V={ch['V']}): "
          f"Vp={Vp:.2f} V, Te={Te:.2f} eV, ne={ne:.3e} cm^-3")

for t in incomplete:
    print(f"tip {t}: BLANK (incomplete I+V pair)")

## 9. Compare two runs — what changed in the setup?

`compare_runs(a, b)` parses each file's hand-written `description` attribute and
diffs the experimental setup, tolerant of formatting drift (`800G` vs `800 G` is
*not* a difference). Set `ifn_b` to a second run; the diff classifies each
setting as **changed** (present on both, differing) or **added/removed**
(present on one side), ranked most-significant first (B-field, gas, bias, ...).
`diff.summary()` collapses it all to one line for a plot title or banner.

In [3]:
from data_analysis.io import compare_runs

# `ifn` (cell 1) is run A; set this to the run you want to compare it against.
ifn_b = r"D:\data\LAPD\jun2026-jia\01-He-800G-bias40V-LP-p29-line_2026-06-10.hdf5"

diff = compare_runs(ifn, ifn_b)

print(f"A: {diff.label_a}")
print(f"B: {diff.label_b}")
print(f"\nMain difference(s): {diff.summary()}\n")

# Full breakdown (raw operator text, not the normalized compare form).
if diff.changed:
    print("Changed (path: A -> B):")
    for path, raw_a, raw_b in diff.changed:
        print(f"  {path}: {raw_a!r} -> {raw_b!r}")
if diff.only_in_a:
    print("\nOnly in A:")
    for path, raw in diff.only_in_a:
        print(f"  {path}: {raw!r}")
if diff.only_in_b:
    print("\nOnly in B:")
    for path, raw in diff.only_in_b:
        print(f"  {path}: {raw!r}")
if not diff:
    print("Setups are identical (no differences in the description).")

A: 00-He-800G-bias40V-LP-p29-line_2026-06-10.hdf5
B: 01-He-800G-bias40V-LP-p29-line_2026-06-10.hdf5

Main difference(s): +gas; -gas


Only in A:
  plasma setting / helium backside pressure 40 psi, puff voltage 75v for 24ms west+east: None

Only in B:
  plasma setting / helium backside pressure 40 psi, puff voltage 75v for 25ms west+east: None
